Goal: Create planet oder and delivery to GEE using the new Planet Python SDK (v2)
reference：notebooks/jupyter-notebooks/gee-integration/gee-integration.ipynb  
https://github.com/planetlabs/notebooks/blob/c1bb38018b1527aefd26c54d5c7aabdc794ffbef/jupyter-notebooks/gee-integration/gee-integration.ipynb#L9

In [8]:
# !pip install planet
# !pip install -U planet 
# In Jupyter Notebook, you can run shell commands (like you would in your terminal or command prompt) by prefixing them with !
# ! pip show planet
# !where python  # Windows
# !pip show planet
# conda install -n planet_env ipykernel --update-deps --force-reinstall


# Installing the package in a specific conda environment from the command line (not in Jupyter Notebook)
# # Install planet
# "C:\Users\pc\.conda\envs\geo_env2\python.exe" -m pip install planet

# # Install ipykernel and register it
# "C:\Users\pc\.conda\envs\geo_env2\python.exe" -m pip install ipykernel
# "C:\Users\pc\.conda\envs\geo_env2\python.exe" -m ipykernel install --user --name geo_env2 --display-name "geo_env2"


In [9]:
import os
import json
import requests 
from requests.auth import HTTPBasicAuth
import planet

print(planet.__version__)

#planet SDK
from planet import Auth, reporting, Session, OrdersClient, order_request, data_filter

3.5.0


In [ ]:
from dotenv import load_dotenv

load_dotenv(r"D:\natcap\mongolia-mining\.env")

API_KEY = os.getenv("PLANET_API_KEY")
print("Key loaded:", API_KEY is not None)  # confirm without exposing key


Key loaded: True


In [ ]:
# if your Planet API Key is not set as an environment variable, you can paste it below
if 'PL_API_KEY' in os.environ:
    API_KEY = os.environ['PLANET_API_KEY']
    print("Key loaded")
else:
    API_KEY = 'paste_your_API_KEY_here' 
    os.environ['PL_API_KEY'] = API_KEY

client = Auth.from_key(API_KEY)

Key loaded


# 1. Define data to be ordered

Search and filter images from [Planet Exporer](https://www.planet.com/explorer/) -> select the high quility images and copy + paste image ids to the list below. 

For filters: I mainly focus on 
- Upload an AOI for better visual inspection 
- Area coverage: 70 - 100%
- Cloud cover: 0 - 10%
- Date: 7/1 - 10/1 for each year

In [4]:
# The area of interest (AOI) defined as a polygon
# taylor_aoi = {
#     "type":
#     "Polygon",
#     "coordinates": [[[-106.6388064,38.88286528],[-106.60837564,38.88286528],[-106.60837564,38.92908856],[-106.6388064,38.92908856],[-106.6388064,38.88286528]]]
# }


# The item IDs we wish to order
images = [
    # '20160901_030847_0e16', '20160901_030848_0e16',
    '20160901_030849_0e16',
    
    # '20170905_031709_1041', 
    # '20170905_031710_1041', 
    # '20170905_031711_1041',

    # '20180907_032954_1014', '20180907_032955_1014',
    '20180907_032956_1014',

    # '20190619_034839_53_1064', 
    '20190825_033805_1014', '20190825_033804_1014', '20190825_033803_1014',

    # '20200902_034004_1006', '20200902_034006_1006', '20200902_034007_1006',
    # '20210906_040132_85_2426', '20210906_040135_15_2426',

    # '20220923_031748_05_2251',
    '20220923_031750_16_2251',

    # '20230902_030946_82_24af', '20230902_030949_09_24af',
    # '20240911_040629_70_24f3',
    ]


# 2. Define cloud delivery location

In [5]:
# Google Earth Engine configuration
cloud_config = planet.order_request.google_earth_engine(
    project='gee-planet-natcap', 
    collection='planet_Mongolia',)
# Order delivery configuration
delivery_config = planet.order_request.delivery(cloud_config=cloud_config)

# 3. Building an order request, where we specify the products we wish to order. 

In [6]:
# Product description for the order request
data_products = [
    planet.order_request.product(item_ids = images,
                                 product_bundle = 'analytic_sr_udm2',
                                 item_type = 'PSScene')
]

# Build the order request
order = planet.order_request.build_request(name = 'PolygonA',
                                           products = data_products,
                                           delivery = delivery_config)

print(order)

{'name': 'PolygonA', 'products': [{'item_ids': ['20160901_030849_0e16', '20180907_032956_1014', '20190825_033805_1014', '20190825_033804_1014', '20190825_033803_1014', '20220923_031750_16_2251'], 'item_type': 'PSScene', 'product_bundle': 'analytic_sr_udm2'}], 'delivery': {'google_earth_engine': {'project': 'gee-planet-natcap', 'collection': 'planet_Mongolia'}}}


# 4. Create and deliver the order

In [7]:
async def create_and_deliver_order(order_request, client):
    '''Create and deliver an order.

    Parameters:
        order_request: An order request
        client: An Order client object
    '''
    with planet.reporting.StateBar(state='creating') as reporter:
        # Place an order to the Orders API
        order = await client.create_order(order_request)
        reporter.update(state='created', order_id=order['id'])
        # Wait while the order is being completed
        await client.wait(order['id'],
                          callback=reporter.update_state,
                          max_attempts=0)

    # Grab the details of the orders
    order_details = await client.get_order(order_id=order['id'])

    return order_details

In [8]:
async with planet.Session() as ps:
    # The Orders API client
    client = ps.client('orders')
    # Create the order and deliver it to GEE
    order_details = await create_and_deliver_order(order, client)

41:41 - order 2a038bdf-4f5a-49e2-96c7-7aa18bdad961 - state: success


In [9]:
print(order_details['_links']['results'][0])

{'delivery': 'success', 'expires_at': '0001-01-01T00:00:00.000Z', 'name': '20190825_033803_1014_3B_AnalyticMS_SR.tif'}
